# Netflix Dataset — Exploratory Data Analysis
**Dataset:** Netflix Titles  
**Total Rows:** 8,807 | **Columns:** 12  
**Goal:** Understand what kind of content Netflix has, who it's for, and how it has grown.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13

print('All libraries imported successfully!')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('netflix_titles.csv')
df.head()

## 3. Dataset Overview

In [ ]:
print('First 5 rows:')
df.head()

In [ ]:
print('Last 5 rows:')
df.tail()

In [ ]:
print('Shape (rows, columns):', df.shape)

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
print('Column names:', df.columns.tolist())
print()
print('Data types:')
print(df.dtypes)

## 4. Data Cleaning

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# director and cast are missing a lot — that's normal (not all titles have listed info)
# We fill them with 'Unknown' so they don't break anything
df['director'].fillna('Unknown', inplace=True)
df['cast'].fillna('Unknown', inplace=True)
df['country'].fillna('Unknown', inplace=True)

# For rating and duration — only a handful missing, safe to drop those rows
df.dropna(subset=['rating', 'duration', 'date_added'], inplace=True)

print('Missing values after cleaning:')
print(df.isnull().sum())

In [ ]:
# Check duplicates
print('Duplicate rows:', df.duplicated().sum())
# None found — no action needed

In [ ]:
# Fix data types
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), format='%B %d, %Y', errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

# Some rows in 'rating' accidentally have duration values — fix them
wrong_ratings = df['rating'].isin(['74 min', '84 min', '66 min'])
df.loc[wrong_ratings, 'duration'] = df.loc[wrong_ratings, 'rating']
df.loc[wrong_ratings, 'rating'] = np.nan
df.dropna(subset=['rating'], inplace=True)

# Extract numeric duration for movies
df['duration_min'] = df['duration'].str.extract(r'(\d+)').astype(float)

print('Data types fixed!')
print(df[['date_added', 'year_added', 'duration_min']].head())

## 5. Outlier Detection

In [ ]:
# We check outliers only for movies (duration in minutes)
movies = df[df['type'] == 'Movie'].copy()

plt.figure(figsize=(8, 4))
sns.boxplot(x=movies['duration_min'], color='tomato')
plt.title('Movie Duration — Boxplot (before removing outliers)')
plt.xlabel('Duration (minutes)')
plt.tight_layout()
plt.show()

In [ ]:
# Handle outliers using IQR method
Q1 = movies['duration_min'].quantile(0.25)
Q3 = movies['duration_min'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f'Lower bound: {lower} min | Upper bound: {upper} min')
print(f'Outliers found: {((movies["duration_min"] < lower) | (movies["duration_min"] > upper)).sum()}')

movies_clean = movies[(movies['duration_min'] >= lower) & (movies['duration_min'] <= upper)]
print(f'Movies after removing outliers: {len(movies_clean)}')

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=movies_clean['duration_min'], color='mediumseagreen')
plt.title('Movie Duration — Boxplot (after removing outliers)')
plt.xlabel('Duration (minutes)')
plt.tight_layout()
plt.show()

## 6. Univariate Analysis

In [ ]:
# Movies vs TV Shows
type_counts = df['type'].value_counts()

plt.figure(figsize=(6, 5))
plt.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%',
        colors=['#E50914', '#564d4d'], startangle=90,
        textprops={'fontsize': 12})
plt.title('Movies vs TV Shows on Netflix')
plt.tight_layout()
plt.show()

In [ ]:
# Content added per year
yearly = df['year_added'].value_counts().sort_index()

plt.figure(figsize=(11, 5))
sns.barplot(x=yearly.index.astype(int), y=yearly.values, palette='Reds_r')
plt.title('How Many Titles Netflix Added Each Year')
plt.xlabel('Year')
plt.ylabel('Number of Titles')
plt.tight_layout()
plt.show()

In [ ]:
# Top content ratings
rating_counts = df['rating'].value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=rating_counts.index, y=rating_counts.values, palette='magma')
plt.title('Most Common Content Ratings on Netflix')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 countries
country_counts = df[df['country'] != 'Unknown']['country'].value_counts().head(10)

plt.figure(figsize=(11, 5))
sns.barplot(x=country_counts.values, y=country_counts.index, palette='crest')
plt.title('Top 10 Countries by Number of Titles')
plt.xlabel('Count')
plt.ylabel('Country')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of movie durations
plt.figure(figsize=(10, 5))
sns.histplot(movies_clean['duration_min'], bins=40, kde=True, color='#E50914')
plt.title('Distribution of Movie Durations (minutes)')
plt.xlabel('Duration (min)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Top genres
genres = df['listed_in'].str.split(', ').explode()
top_genres = genres.value_counts().head(12)

plt.figure(figsize=(11, 5))
sns.barplot(x=top_genres.values, y=top_genres.index, palette='flare')
plt.title('Top 12 Genres on Netflix')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

## 7. Bivariate Analysis

In [ ]:
# Movies vs TV Shows added each year
type_year = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)

type_year.plot(kind='bar', figsize=(12, 5), color=['#E50914', '#564d4d'])
plt.title('Movies vs TV Shows Added Per Year')
plt.xlabel('Year')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(title='Type')
plt.tight_layout()
plt.show()

In [ ]:
# Rating distribution for Movies vs TV Shows
top_ratings = df['rating'].value_counts().head(8).index
df_top_ratings = df[df['rating'].isin(top_ratings)]

plt.figure(figsize=(12, 5))
sns.countplot(data=df_top_ratings, x='rating', hue='type',
              order=top_ratings, palette=['#E50914', '#564d4d'])
plt.title('Content Ratings: Movies vs TV Shows')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.legend(title='Type')
plt.tight_layout()
plt.show()

In [ ]:
# Movie release year vs when it was added (how old was the content when added?)
movies_subset = movies_clean.dropna(subset=['year_added', 'release_year']).copy()
movies_subset['age_when_added'] = movies_subset['year_added'] - movies_subset['release_year']

plt.figure(figsize=(10, 5))
sns.scatterplot(data=movies_subset.sample(1000, random_state=42),
                x='release_year', y='year_added',
                hue='age_when_added', palette='coolwarm', alpha=0.6)
plt.title('When Was the Movie Released vs When It Was Added to Netflix')
plt.xlabel('Release Year')
plt.ylabel('Year Added to Netflix')
plt.tight_layout()
plt.show()

In [ ]:
# Month-wise content additions (which month does Netflix add the most content?)
month_counts = df['month_added'].value_counts().sort_index()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

plt.figure(figsize=(11, 5))
sns.barplot(x=month_names, y=month_counts.values, palette='Reds_r')
plt.title('Which Month Does Netflix Add the Most Content?')
plt.xlabel('Month')
plt.ylabel('Titles Added')
plt.tight_layout()
plt.show()

## 8. Correlation Analysis

In [ ]:
# We'll build a correlation matrix using numeric + encoded features
corr_df = df[['release_year', 'year_added', 'month_added', 'duration_min']].copy()
corr_df['is_movie'] = (df['type'] == 'Movie').astype(int)

# Age of content when added
corr_df['age_when_added'] = corr_df['year_added'] - corr_df['release_year']

corr_matrix = corr_df.dropna().corr()

plt.figure(figsize=(9, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# What's strongly correlated?
corr_pairs = corr_matrix.unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs < 1.0]  # remove self-correlations

print('Most Positively Correlated:')
print(corr_pairs.head(5))
print()
print('Most Negatively Correlated:')
print(corr_pairs.tail(5))

**Key Observations:**
- `year_added` and `release_year` are positively correlated — Netflix tends to add recent content
- `age_when_added` and `release_year` are negatively correlated — older movies waited longer to be added
- `is_movie` and `duration_min` are positively correlated — movies obviously have higher minutes than TV shows counted in seasons

## 9. Feature Engineering

In [ ]:
# Feature 1: Content Age Category — how old was the content when added?
df['age_when_added'] = df['year_added'] - df['release_year']

def age_bucket(age):
    if pd.isna(age):
        return 'Unknown'
    elif age <= 1:
        return 'New Release (0-1 yr)'
    elif age <= 5:
        return 'Recent (2-5 yrs)'
    elif age <= 15:
        return 'Older (6-15 yrs)'
    else:
        return 'Classic (15+ yrs)'

df['content_age_category'] = df['age_when_added'].apply(age_bucket)

print('Content Age Category Distribution:')
print(df['content_age_category'].value_counts())

plt.figure(figsize=(8, 5))
order = ['New Release (0-1 yr)', 'Recent (2-5 yrs)', 'Older (6-15 yrs)', 'Classic (15+ yrs)', 'Unknown']
sns.countplot(data=df, x='content_age_category', order=order,
              hue='type', palette=['#E50914', '#564d4d'])
plt.title('How Old Was the Content When Netflix Added It?')
plt.xlabel('Age Category')
plt.ylabel('Count')
plt.xticks(rotation=15)
plt.legend(title='Type')
plt.tight_layout()
plt.show()

In [ ]:
# Feature 2: Is content for adults? (based on rating)
adult_ratings = ['TV-MA', 'R', 'NC-17']
df['is_adult_content'] = df['rating'].isin(adult_ratings)

print('Adult vs Non-Adult Content:')
print(df['is_adult_content'].value_counts())

In [ ]:
# Feature 3: Number of genres per title
df['genre_count'] = df['listed_in'].str.split(', ').apply(len)

print('Average genres per title:', df['genre_count'].mean().round(2))
print(df['genre_count'].value_counts().sort_index())

## 10. Business Insights

In [ ]:
insights = [
    "1. Netflix is heavily movie-focused — 70% of content is movies. TV shows are growing but still a smaller portion.",
    "2. The US dominates with 2,818 titles. India is second with 972. Netflix should keep investing in Indian content given the massive market.",
    "3. TV-MA (adult content) is the most common rating — Netflix is targeting adults, not families.",
    "4. Netflix had explosive growth between 2016-2019. Content additions slowed after 2019, possibly due to budget pressure or COVID.",
    "5. Most movies are 87-114 minutes long — the sweet spot for comfortable viewing without commitment fatigue.",
    "6. International Movies and Dramas dominate the genre chart — global drama content is clearly a strategic priority.",
    "7. January, October, and December see the most content additions — Netflix loads up before New Year and holiday season.",
    "8. A large chunk of content is 'Recent (2-5 years old)' when added — Netflix prefers semi-recent content, not fresh theatrical releases.",
    "9. Kid-friendly content (TV-Y, TV-G, TV-PG) is a small portion. Netflix could grow its family segment to compete with Disney+.",
    "10. Many titles span 3+ genres — Netflix intentionally cross-categorizes content to appear in more search results and recommendations."
]

for insight in insights:
    print(insight)
    print()

## 11. Conclusion

In [ ]:
conclusion = """
CONCLUSION
----------
Netflix's catalog of 8,800+ titles tells a clear story:

- The platform is built for adults who want movies. 70% of content is films, and the most common 
  rating is TV-MA. Family content is noticeably underrepresented.

- The US is the top producer, but India is rising fast — and this likely reflects where Netflix 
  sees its next wave of subscribers.

- Netflix's biggest growth happened between 2016-2019. The platform built its library quickly 
  during this period, and that library now serves as its biggest competitive advantage.

- Content is typically added 2-5 years after release, not right out of theatres. This means 
  Netflix acts more like a library than a first-run cinema.

- Dramas and International content are clearly the bread and butter — this is what subscribers 
  binge the most, and what Netflix keeps adding the most of.

For Netflix to keep growing, it should focus on:
  → More family-friendly content to compete with Disney+
  → Deeper investment in India, Korea, and other high-growth markets
  → Balancing its adult-heavy catalog with content for younger demographics
"""
print(conclusion)

---
### One-Page Business Brief

| | |
|---|---|
| **Objective** | Analyze Netflix's content library to find patterns in content type, country, ratings, and growth |
| **Dataset** | Netflix Titles — 8,807 rows, 12 columns |
| **Key Findings** | 70% movies, US dominates, adult content is most common, growth peaked 2016–2019 |
| **Recommendations** | Invest in family content, expand Indian/Korean libraries, add more recent releases |

**Prepared by:** [Your Name]  
**Date:** June 2025